# 06 · Running crawlers as a platform

Define a spider as data, run it as a job, schedule it with cron, watch live metrics and receive alerts.

*Definisikan spider sebagai data, jalankan sebagai job, jadwalkan dengan cron, pantau metrik, dan terima alert.*

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil.*

In [ ]:
// Packages / Paket
#r "nuget: Gravicode.ScrapyNet, 0.1.0"
#r "nuget: Gravicode.ScrapyNet.Sandbox, 0.1.0"
#r "nuget: Gravicode.ScrapyNet.Platform, 0.1.0"

// Before the packages are on nuget.org, point at a local build instead:
//   dotnet pack ScrapyNet.slnx -c Release -o artifacts
// and add this line above the #r lines:  #i "nuget: ../artifacts"

In [ ]:
using ScrapyNet;
using ScrapyNet.Platform;
using ScrapyNet.Sandbox;

var site = await SandboxSite.StartAsync();
var catalog = new SpiderCatalog();
catalog.Create(new SpiderDefinition
{
    Id = "books",
    Name = "books",
    Target = new TargetConfig { StartUrls = [site.Url("/books/")] },
    Rules = new ScrapingRules
    {
        DetailLinks = ["article.product_pod h3 a"],
        FollowLinks = ["li.next a"],
        Fields = new()
        {
            ["title"] = new FieldRule { Css = "h1::text", Processors = ["clean"] },
            ["price"] = new FieldRule { Css = "p.price_color::text", Processors = ["price"] },
        },
    },
});
display(catalog.List().Select(d => $"{d.Id} v{d.Version} declarative={d.IsDeclarative}"));

In [ ]:
var jobs = new JobManager(catalog, new JobManagerOptions
{
    BaseSettings = () => { var s = new Settings(); s.Set(SettingKeys.LogLevel, "WARNING"); return s; },
});
var monitor = new CrawlMonitor(jobs);
var alerts = new List<Alert>();
new NotificationDispatcher(jobs).Route(new CallbackNotifier(a => { alerts.Add(a); return Task.CompletedTask; }), AlertSeverity.Info);

var job = await jobs.WaitAsync(jobs.RunNow("books").Id);
display($"{job.Status}: {job.ItemCount} items, {job.ResponseCount} responses in {job.Duration?.TotalSeconds:0.00}s");
display(alerts);

In [ ]:
var cron = new CronExpression(Schedules.Weekly(DayOfWeek.Monday, 6, 30));
display(cron.GetOccurrences(DateTimeOffset.UtcNow, 3).Select(t => t.ToString("ddd yyyy-MM-dd HH:mm")));

var metrics = monitor.Get(job.Id)!;
display($"requests={metrics.Requests} responses={metrics.Responses} items={metrics.Items} errorRate={metrics.ErrorRate:P1}");
await site.DisposeAsync();